# SE3082 Lab 10 – CUDA: timing, errors, 2D and 3D grids
Student ID: IT24100532

Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**.

In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv

## Exercise 1 – Timing a GPU program
### (a) The timed program

In [ ]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // initialize timer
    dStart = cpuSecond();

    // Launch add() kernel on GPU with N blocks
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock

    dElaps = cpuSecond() - dStart;

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

### Experiment – remove `cudaDeviceSynchronize()`

In [ ]:
%%writefile ex1_nosync.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // initialize timer
    dStart = cpuSecond();

    // Launch add() kernel on GPU with N blocks
    add<<<N, 1>>>(d_a, d_b, d_c);
    // EXPERIMENT: cudaDeviceSynchronize() removed - the clock stops right after the launch

    dElaps = cpuSecond() - dStart;

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex1_nosync.cu -o ex1_nosync && ./ex1_nosync

**What happens:** `Time taken` becomes much smaller, because a kernel launch is asynchronous: the CPU starts the kernel and stops the clock immediately, so only the launch is timed, not the work. The answer still verifies correctly (0 mismatches), because the next `cudaMemcpy` (device to host) waits for the kernel to finish before it copies the result.

### (b) Lab 9 programs with the timer (run each one 3 times and fill in the table in `answers.md`)

In [ ]:
%%writefile ex3_blocks_timed.cu
// Lab 10 Exercise 1(b): Lab 9 ex3_blocks with the CPU timer added around the kernel.
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    double dStart, dElaps;
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel on GPU with N blocks, 1 thread per block
    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock
    dElaps = cpuSecond() - dStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex3_blocks_timed.cu -o ex3_blocks_timed && ./ex3_blocks_timed

In [ ]:
%%writefile ex4_threads_timed.cu
// Lab 10 Exercise 1(b): Lab 9 ex4_threads with the CPU timer added around the kernel.
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    double dStart, dElaps;
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch addT() kernel on GPU with 1 block of N threads
    dStart = cpuSecond();
    addT<<<1, N>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock
    dElaps = cpuSecond() - dStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex4_threads_timed.cu -o ex4_threads_timed && ./ex4_threads_timed

In [ ]:
%%writefile ex5_vecmul_timed.cu
// Lab 10 Exercise 1(b): Lab 9 ex5_vecmul with the CPU timer added around the kernel.
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 10000000              // 10 million elements
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // TODO 1: global index of this thread
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: only compute if i is inside the array. The last block has 384 extra
    // threads (19,532 x 512 = 10,000,384), and without this check they would read and
    // write past the end of the arrays.
    if (i < n)
        c[i] = a[i] * b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    double dStart, dElaps;
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate the input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // TODO 3: number of blocks needed to cover N elements (round UP)
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    // TODO 4: launch vecMul with <<<blocks, THREADS_PER_BLOCK>>>
    dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock
    dElaps = cpuSecond() - dStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 5: copy the result d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 6: print the last 1000 results as "index) a x b = c"
    for (int i = N - 1000; i < N; i++)
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    // TODO 7: verify all N results on the CPU and print the number of mismatches
    int errors = 0;
    for (int i = 0; i < N; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex5_vecmul_timed.cu -o ex5_vecmul_timed && ./ex5_vecmul_timed

In [ ]:
%%writefile ex6_matrix_timed.cu
// Lab 10 Exercise 1(b): Lab 9 ex6_matrix with the CPU timer added around the kernel.
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define ROWS 10000
#define COLS 10000

// Element-wise product: C[r][c] = A[r][c] * B[r][c]  (NOT matrix multiplication)
__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    // TODO 1: row comes from the y dimension, col from the x dimension
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: only compute if BOTH row and col are inside the matrix
    if (row < rows && col < cols) {
        // TODO 3: convert (row, col) into a flat index: row * cols + col
        size_t idx = (size_t)row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    double dStart, dElaps;
    int *a, *b, *c;             // host copies (flat arrays, row-major)
    int *d_a, *d_b, *d_c;       // device copies
    size_t count = (size_t)ROWS * COLS;
    size_t size  = count * sizeof(int);         // 400 MB per matrix

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 32 x 16 = 512 threads per block
    dim3 threadsPerBlock(32, 16);
    // TODO 4: blocks needed in x (columns) and in y (rows), rounding UP
    dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    // TODO 5: launch mulElem with <<<numBlocks, threadsPerBlock>>>
    dStart = cpuSecond();
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);
    cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock
    dElaps = cpuSecond() - dStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 6: copy d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 7: print C[9999][9000] ... C[9999][9999]  (the last 1000 elements)
    int r = ROWS - 1;
    for (int col = COLS - 1000; col < COLS; col++) {
        size_t idx = (size_t)r * COLS + col;
        printf("C[%d][%d] = %d x %d = %d\n", r, col, a[idx], b[idx], c[idx]);
    }

    // TODO 8: verify all elements on the CPU and print the number of mismatches
    size_t errors = 0;
    for (size_t i = 0; i < count; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %zu mismatches out of %zu\n", errors, count);

    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex6_matrix_timed.cu -o ex6_matrix_timed && ./ex6_matrix_timed

### (c) Kernel time vs whole copy-in → compute → copy-out time (100 million elements)

In [ ]:
%%writefile ex5_vecmul_100M_timed.cu
// Lab 10 Exercise 1(c): timed vector multiply with 100 million elements.
// Prints the kernel-only time and the whole copy-in -> compute -> copy-out time.
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 100000000             // 100 million elements (about 400 MB per array)
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // TODO 1: global index of this thread
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: only compute if i is inside the array. The last block has 384 extra
    // threads (19,532 x 512 = 10,000,384), and without this check they would read and
    // write past the end of the arrays.
    if (i < n)
        c[i] = a[i] * b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    double dStart, dElaps, wStart, wElaps;
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate the input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Whole sequence starts before the first host-to-device copy
    wStart = cpuSecond();

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // TODO 3: number of blocks needed to cover N elements (round UP)
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    // TODO 4: launch vecMul with <<<blocks, THREADS_PER_BLOCK>>>
    dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock
    dElaps = cpuSecond() - dStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 5: copy the result d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
    wElaps = cpuSecond() - wStart;   // ...and stops after the last device-to-host copy

    // print only the last 5 results (100 million is too many to print)
    for (int i = N - 5; i < N; i++)
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    // TODO 7: verify all N results on the CPU and print the number of mismatches
    int errors = 0;
    for (int i = 0; i < N; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Kernel only        - %.7f s\n", dElaps);
    printf("Copy in + kernel + copy out - %.7f s\n", wElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex5_vecmul_100M_timed.cu -o ex5_vecmul_100M_timed && ./ex5_vecmul_100M_timed

**Think:** the whole sequence takes much longer than the kernel alone, because 800 MB has to be copied to the GPU and 400 MB copied back over the PCIe bus, while the multiplication itself is very simple. A GPU is worth using when there is a lot of computation for each byte that has to be copied.

## Exercise 2 – Catching errors with `CHECK()`

In [ ]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// Checks the value returned by a CUDA runtime call. On failure it prints the
// file, line number, error code and reason, then stops the program.
#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // TODO 1: wrap each of the three cudaMalloc calls in CHECK(...)
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // TODO 2: wrap both host-to-device cudaMemcpy calls in CHECK(...)
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    // TODO 3: add CHECK(cudaGetLastError()); here, and wrap cudaDeviceSynchronize()
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    // TODO 4: wrap the device-to-host cudaMemcpy in CHECK(...)
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    // TODO 5: wrap the three cudaFree calls in CHECK(...)
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check

### Experiment A – too many threads (`<<<N, 2048>>>`), caught by `CHECK(cudaGetLastError())`

In [ ]:
%%writefile ex2_check_expA.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// Checks the value returned by a CUDA runtime call. On failure it prints the
// file, line number, error code and reason, then stops the program.
#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // TODO 1: wrap each of the three cudaMalloc calls in CHECK(...)
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // TODO 2: wrap both host-to-device cudaMemcpy calls in CHECK(...)
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dStart = cpuSecond();
    add<<<N, 2048>>>(d_a, d_b, d_c);   // EXPERIMENT A: 2048 threads per block (limit is 1024)
    // TODO 3: add CHECK(cudaGetLastError()); here, and wrap cudaDeviceSynchronize()
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    // TODO 4: wrap the device-to-host cudaMemcpy in CHECK(...)
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    // TODO 5: wrap the three cudaFree calls in CHECK(...)
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex2_check_expA.cu -o ex2_check_expA && ./ex2_check_expA

### Experiment B – impossible allocation (1 petabyte), caught by `CHECK(cudaMalloc(...))`

In [ ]:
%%writefile ex2_check_expB.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// Checks the value returned by a CUDA runtime call. On failure it prints the
// file, line number, error code and reason, then stops the program.
#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // TODO 1: wrap each of the three cudaMalloc calls in CHECK(...)
    CHECK(cudaMalloc((void **)&d_a, (size_t)1 << 50));   // EXPERIMENT B: one petabyte
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // TODO 2: wrap both host-to-device cudaMemcpy calls in CHECK(...)
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    // TODO 3: add CHECK(cudaGetLastError()); here, and wrap cudaDeviceSynchronize()
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    // TODO 4: wrap the device-to-host cudaMemcpy in CHECK(...)
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    // TODO 5: wrap the three cudaFree calls in CHECK(...)
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex2_check_expB.cu -o ex2_check_expB && ./ex2_check_expB

### Experiment C – Experiment A on the plain Exercise 1 program (no `CHECK`)

In [ ]:
%%writefile ex1_expC.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // initialize timer
    dStart = cpuSecond();

    // Launch add() kernel on GPU with N blocks
    add<<<N, 2048>>>(d_a, d_b, d_c);   // EXPERIMENT C: too many threads, and no CHECK()
    cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock

    dElaps = cpuSecond() - dStart;

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex1_expC.cu -o ex1_expC && ./ex1_expC

**Experiment C:** nothing reports an error. The kernel never runs, so the program prints wrong results (`c` was never written) and only the verification line (512 mismatches) shows that something went wrong. Without the verification loop the wrong answer would look normal.

Which row of the table each experiment tests: A = a wrong kernel launch (caught by `CHECK(cudaGetLastError())`), B = a CUDA call that returns an error (`CHECK(cudaMalloc(...))`), C = the same launch error with no check at all.

## Exercise 3 – Scaling a picture with a 2D grid (`PictureKernel`)

**Predict:** pixels with an original value of 128 or more become 256+ when doubled, so they are capped at 255. That is 128 of the 256 possible values, so about **half** of the 480,000 pixels (about 240,000).

In [ ]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define WIDTH  800              // columns
#define HEIGHT 600              // rows

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Kernel from the lecture slide "Source Code of a PictureKernel", adapted
// for 8-bit pixels: scale every pixel value by 2.0, but never exceed 255.
__global__ void PictureKernel(unsigned char *d_Pin, unsigned char *d_Pout,
                              int height, int width) {
    // Calculate the row # of the d_Pin and d_Pout element
    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    // Calculate the column # of the d_Pin and d_Pout element
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    // each thread computes one element of d_Pout if in range
    if ((Row < height) && (Col < width)) {
        int v = 2 * d_Pin[Row * width + Col];
        d_Pout[Row * width + Col] = (v > 255) ? 255 : v;
    }
}

int main(void) {
    size_t count = (size_t)WIDTH * HEIGHT;      // ONE flat array: 480,000 pixels
    size_t size  = count * sizeof(unsigned char);
    unsigned char *h_in, *h_out;                // host: original and updated picture
    unsigned char *d_in, *d_out;                // device copies
    double dStart, dElaps;

    h_in  = (unsigned char *)malloc(size);
    h_out = (unsigned char *)malloc(size);
    CHECK(cudaMalloc((void **)&d_in,  size));
    CHECK(cudaMalloc((void **)&d_out, size));

    // TODO 1: give every pixel of h_in a random value between 0 and 255
    for (size_t i = 0; i < count; i++)
        h_in[i] = rand() % 256;

    CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));

    // TODO 2: 16 threads per block in BOTH x and y
    dim3 threadsPerBlock(16, 16);
    // TODO 3: blocks needed in x (columns) and in y (rows), rounding UP
    dim3 numBlocks((WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %d x %d blocks of %d x %d threads\n",
           numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y);

    dStart = cpuSecond();
    // TODO 4: launch PictureKernel with <<<numBlocks, threadsPerBlock>>>
    PictureKernel<<<numBlocks, threadsPerBlock>>>(d_in, d_out, HEIGHT, WIDTH);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));

    // TODO 5: print the top-left 50 x 50 portion: first the original values,
    //         then the updated values
    printf("\nOriginal picture, top-left 50 x 50:\n");
    for (int row = 0; row < 50; row++) {
        for (int col = 0; col < 50; col++)
            printf("%3d ", h_in[row * WIDTH + col]);
        printf("\n");
    }
    printf("\nUpdated picture, top-left 50 x 50:\n");
    for (int row = 0; row < 50; row++) {
        for (int col = 0; col < 50; col++)
            printf("%3d ", h_out[row * WIDTH + col]);
        printf("\n");
    }

    // TODO 6: check every pixel on the CPU (expected = 2 x original, capped at 255),
    //         count the mismatches, and count how many pixels were capped
    int errors = 0, capped = 0;
    for (size_t i = 0; i < count; i++) {
        int expected = 2 * h_in[i];
        if (expected > 255) { expected = 255; capped++; }
        if (h_out[i] != expected) errors++;
    }
    printf("\nVerification: %d mismatches out of %zu\n", errors, count);
    printf("Pixels capped at 255: %d\n", capped);

    printf("Time taken - %.7f\n", dElaps);

    free(h_in); free(h_out);
    CHECK(cudaFree(d_in)); CHECK(cudaFree(d_out));
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture

## Exercise 4 – Multiplying two 3D matrices element by element (3D grid)

In [ ]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define NX 500                  // x: columns
#define NY 500                  // y: rows
#define NZ 500                  // z: planes (depth)

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Element-wise product: C[z][y][x] = A[z][y][x] * B[z][y][x]
__global__ void mulElem3D(int *a, int *b, int *c, int nx, int ny, int nz) {
    // TODO 1: the x, y and z coordinates of this thread
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    // TODO 2: only compute if x, y AND z are all inside the volume
    if (x < nx && y < ny && z < nz) {
        // TODO 3: flat index of (x, y, z):  (z * ny + y) * nx + x
        size_t idx = ((size_t)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    size_t count = (size_t)NX * NY * NZ;        // 125,000,000 elements
    size_t size  = count * sizeof(int);         // 500 MB per array
    int *a, *b, *c;                             // host copies (flat arrays)
    int *d_a, *d_b, *d_c;                       // device copies
    double dStart, dElaps;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // 8 x 8 x 8 = 512 threads per block
    dim3 threadsPerBlock(8, 8, 8);
    // TODO 4: blocks needed in x, y and z, rounding UP
    dim3 numBlocks((NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
                   (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z);
    printf("Grid: %d x %d x %d blocks\n", numBlocks.x, numBlocks.y, numBlocks.z);

    dStart = cpuSecond();
    // TODO 5: launch mulElem3D with <<<numBlocks, threadsPerBlock>>>
    mulElem3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // TODO 6: print the last 1000 elements of C, in memory order
    //         (C[499][498][0] ... C[499][499][499]) as "C[z][y][x] = a x b = c"
    for (size_t i = count - 1000; i < count; i++) {
        int x = i % NX, y = (i / NX) % NY, z = i / ((size_t)NX * NY);
        printf("C[%d][%d][%d] = %d x %d = %d\n", z, y, x, a[i], b[i], c[i]);
    }

    // TODO 7: check all elements on the CPU and print the number of mismatches
    size_t errors = 0;
    for (size_t i = 0; i < count; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %zu mismatches out of %zu\n", errors, count);

    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume

**Think:** compare the `Time taken` of this kernel (125 million elements) with Lab 9 Exercise 6 (100 million elements, timed in Exercise 1(b)). Each element does the same tiny amount of work, so both kernels are limited by how fast the GPU can read and write memory, not by how the threads are arranged. The 3D kernel should take roughly 1.25× as long, because it has 1.25× as many elements. The arrangement would matter more if neighbouring threads read memory that is far apart (x is the fastest-changing index here, so neighbouring threads read neighbouring memory in both kernels).